In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# 1. Tải và chuẩn hóa dữ liệu
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Biến đổi ảnh từ (32, 32, 3) thành dạng chuỗi (32, 96) để đưa vào LSTM
X_train_seq = X_train.reshape(-1, 32, 32 * 3)
X_test_seq = X_test.reshape(-1, 32, 32 * 3)

# 2. Xây dựng mô hình LSTM
model_cifar = models.Sequential([
    layers.LSTM(128, input_shape=(32, 96), return_sequences=True),
    layers.LSTM(64),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax') # 10 nhãn đầu ra
])

model_cifar.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 3. Huấn luyện mô hình
model_cifar.fit(X_train_seq, y_train, epochs=10, batch_size=64, validation_data=(X_test_seq, y_test))

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Tải dữ liệu giả định từ file npy có sẵn
X = np.load('dogs_vs_cats_photos.npy').astype('float32') / 255.0
y = np.load('dogs_vs_cats_labels.npy')

# Biến đổi dữ liệu sang dạng chuỗi thích hợp với LSTM (200 bước thời gian, mỗi bước có 200 * 3 phần tử)
X_seq = X.reshape(-1, 200, 200 * 3)

X_train, X_test, y_train, y_test = train_test_split(X_seq, y, test_size=0.2, random_state=42)

# 2. Xây dựng mô hình LSTM phân loại nhị phân
model_cat_dog = models.Sequential([
    layers.LSTM(128, input_shape=(200, 600)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid') # Đầu ra nhị phân
])

model_cat_dog.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_cat_dog.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

In [ ]:
# 1. Tải bộ dữ liệu Fashion-MNIST
fashion_mnist = tf.keras.datasets.fashion_mnist
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# Chuẩn hóa ảnh về khoảng [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# 2. Xây dựng mô hình LSTM (28 bước thời gian, mỗi bước có 28 giá trị pixel)
model_fashion = models.Sequential([
    layers.LSTM(128, input_shape=(28, 28)),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_fashion.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_fashion.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_test, y_test))

In [ ]:
# Giả sử đã có file dữ liệu khuôn mặt face_x.npy và nhãn face_y.npy (Nam: 0, Nữ: 1)
# X_faces = np.load('face_x.npy').astype('float32') / 255.0
# y_faces = np.load('face_y.npy')

# Khởi tạo dữ liệu giả lập mẫu để chạy được code:
X_faces = np.random.rand(1000, 64, 64, 3).astype('float32')
y_faces = np.random.randint(0, 2, size=(1000, 1))

# Chuyển ảnh cấu trúc không gian sang chuỗi thời gian cho LSTM
X_faces_seq = X_faces.reshape(-1, 64, 64 * 3)

X_train, X_test, y_train, y_test = train_test_split(X_faces_seq, y_faces, test_size=0.2, random_state=42)

model_gender = models.Sequential([
    layers.LSTM(64, input_shape=(64, 192)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model_gender.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_gender.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

In [ ]:
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Đoạn trích mẫu Truyện Kiều làm dữ liệu huấn luyện
data_kieu = """
Trăm năm trong cõi người ta
Chữ tài chữ mệnh khéo là ghét nhau
Trải qua một cuộc bể dâu
Những điều trông thấy mà đau đớn lòng
"""

# Khởi tạo Tokenizer để tách từ
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data_kieu])
total_words = len(tokenizer.word_index) + 1

# Tạo các chuỗi n-gram làm dữ liệu huấn luyện
input_sequences = []
for line in data_kieu.strip().split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad các chuỗi để có độ dài bằng nhau
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Chia X (các từ trước) và y (từ tiếp theo)
X_text, y_text = input_sequences[:,:-1], input_sequences[:,-1]
y_text = tf.keras.utils.to_categorical(y_text, num_classes=total_words)

# 2. Xây dựng mô hình LSTM sinh văn bản
model_gen_kieu = models.Sequential([
    Embedding(total_words, 64, input_length=max_sequence_len-1),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model_gen_kieu.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_gen_kieu.fit(X_text, y_text, epochs=100, verbose=0) # Huấn luyện 100 chu kỳ ẩn

# 3. Hàm thử nghiệm sinh câu thơ tiếp theo
def generate_poetry(seed_text, next_words=4):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted = np.argmax(model_gen_kieu.predict(token_list, verbose=0), axis=-1)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

print("Câu 5 - Thử nghiệm sinh thơ:")
print(generate_poetry("Trăm năm trong", next_words=3))

In [ ]:
# Giả lập tập dữ liệu text từ Twitter
twitter_data = [
    "Machine learning is changing the world.",
    "Deep learning with LSTM is very powerful.",
    "Tensorflow makes AI development easy."
]

# Khởi tạo Tokenizer đặc biệt giữ lại dấu chấm câu làm một token riêng biệt
tokenizer_tw = Tokenizer(filters='!"#$%&()*+,-/:;<=>?@[\\]^_`{|}~\t\n')
tokenizer_tw.fit_on_texts(twitter_data)
total_words_tw = len(tokenizer_tw.word_index) + 1

# Tìm token index của dấu chấm câu '.'
dot_index = tokenizer_tw.word_index.get('.')

input_seq_tw = []
for line in twitter_data:
    token_list = tokenizer_tw.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_seq_tw.append(token_list[:i+1])

max_len_tw = max([len(x) for x in input_seq_tw])
input_seq_tw = np.array(pad_sequences(input_seq_tw, maxlen=max_len_tw, padding='pre'))

X_tw, y_tw = input_seq_tw[:,:-1], input_seq_tw[:,-1]
y_tw = tf.keras.utils.to_categorical(y_tw, num_classes=total_words_tw)

# Xây dựng mô hình tương tự câu 5
model_tw = models.Sequential([
    Embedding(total_words_tw, 32, input_length=max_len_tw-1),
    layers.LSTM(64),
    layers.Dense(total_words_tw, activation='softmax')
])

model_tw.compile(loss='categorical_crossentropy', optimizer='adam')
model_tw.fit(X_tw, y_tw, epochs=150, verbose=0)

# Hàm sinh câu tự động dừng khi gặp dấu chấm '.'
def generate_tweet(seed_text, max_limit=15):
    for _ in range(max_limit):
        token_list = tokenizer_tw.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len_tw-1, padding='pre')
        predicted = np.argmax(model_tw.predict(token_list, verbose=0), axis=-1)[0]

        output_word = ""
        for word, index in tokenizer_tw.word_index.items():
            if index == predicted:
                output_word = word
                break

        if output_word == ".":
            seed_text += "."
            break
        else:
            seed_text += " " + output_word

    return seed_text

print("\nCâu 6 - Thử nghiệm sinh tweet hoàn chỉnh câu:")
print(generate_tweet("Machine learning"))